# K1: data bank (trigger bank, real-world test set, CLAP embeddings)

| | |
| --- | --- |
| **Purpose** | Download the training corpora, collect CC0 and CC BY trigger clips from Freesound, lock the 80-clip real-world test set, compute CLAP embeddings for the 4 triggers and 300 phrases, and check 200 generated mixtures with an independent judge (PANNs). |
| **Inputs** | None attached: everything downloads with internet on (ESC-50, FSD50K labels, LibriSpeech, MUSAN, OpenSLR room responses, Freesound previews). Optional: a previous run's output attached as `k1-resume` to continue an interrupted run. |
| **Secrets** | `FREESOUND_API_KEY` (Add-ons > Secrets). |
| **Outputs** | `/kaggle/working/outputs/K1_data_bank/` (manifest, metrics, report, embeddings, data card, test-set lock, 5 samples, 20 spectrograms), zipped as `/kaggle/working/K1_data_bank_outputs.zip`. The trigger bank and test-set audio stay in `/kaggle/working/trigger_bank/` and `/kaggle/working/realworld/` for K2. |
| **Accelerator** | GPU T4 (or P100). The smoke test also runs on CPU. |
| **Internet** | On. |
| **Runtime** | Smoke test about 10 to 20 minutes. Full run about 2 to 4 hours (unverified until the first run). |

Set `SMOKE_TEST = True` first and run all cells. When it prints `SMOKE OK`, set it to `False` and use Save Version > Save & Run All.


## 1. Settings
The only cell you edit: `SMOKE_TEST`.

In [ ]:
SMOKE_TEST = True

REPO_URL = "https://github.com/DarainHyder/anechoic-chamber.git"
REPO_COMMIT = "3e572c0c421bfe18978489d2ccaea4d72c1b8c10"  # the exact code this notebook runs
SEED = 1234

from pathlib import Path

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
RESUME_FROM = INPUT / "k1-resume"  # optional: a previous run's output, to continue it
SCRATCH = Path("/tmp/k1")  # large corpora live here, outside the saved output
REPO_DIR = SCRATCH / "repo"
DATA_DIR = SCRATCH / "data"
BANK_DIR = WORK / "trigger_bank"
REALWORLD_DIR = WORK / "realworld"
STATE_DIR = WORK / "state"
OUT_DIR = WORK / "outputs" / "K1_data_bank"
ZIP_PATH = WORK / "K1_data_bank_outputs"

if REPO_COMMIT == "PENDING_PUSH":
    raise RuntimeError("REPO_COMMIT is not set: use the notebook version from the repo after the code is pushed")
print("smoke test:", SMOKE_TEST, "| commit:", REPO_COMMIT)


## 2. Setup: pinned packages, seeds, GPU, ffmpeg, disk

In [ ]:
import os, random, shutil, subprocess, sys

assert sys.version_info >= (3, 10), f"Python 3.10+ needed, found {sys.version}"
PINNED = ["pyloudnorm==0.2.0", "panns-inference==0.1.1", "soundfile==0.14.0"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINNED], check=True)

import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

HAS_GPU = torch.cuda.is_available()
if not SMOKE_TEST and not HAS_GPU:
    raise RuntimeError("No GPU found. Settings > Accelerator > GPU T4, then restart. (Only the smoke test may run on CPU.)")
DEVICE = "cuda" if HAS_GPU else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if HAS_GPU else "")

if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "install", "-y", "-q", "ffmpeg"], check=True)
assert shutil.which("ffmpeg"), "ffmpeg is required for codec simulation"

SCRATCH.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(SCRATCH).free / 1e9
need_gb = 5 if SMOKE_TEST else 35
print(f"free disk for corpora: {free_gb:.0f} GB (need {need_gb} GB)")
assert free_gb >= need_gb, "not enough disk space for the corpora"


## 3. Code: clone the repo at the pinned commit

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", REPO_COMMIT], check=True)
head = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
assert head == REPO_COMMIT, f"checked out {head}, expected {REPO_COMMIT}"
sys.path.insert(0, str(REPO_DIR))

from synth import download, k1
from synth.freesound import FreesoundClient
from synth.realism import PannsTagger
print("repo ready at", head)


## 4. Resume a previous run, if attached

In [ ]:
# Continue an interrupted run: copy its saved state, bank and test-set audio back.
if RESUME_FROM.exists():
    for name in ("state", "trigger_bank", "realworld"):
        source = RESUME_FROM / name
        if source.exists():
            shutil.copytree(source, WORK / name, dirs_exist_ok=True)
            print("resumed", name, "from", source)
else:
    print("no previous run attached; starting fresh")


## 5. Secrets and models (Freesound, PANNs judge, CLAP)

In [ ]:
from kaggle_secrets import UserSecretsClient
from transformers import ClapModel, ClapProcessor
import transformers

client = FreesoundClient(UserSecretsClient().get_secret("FREESOUND_API_KEY"))
tagger = PannsTagger(device=DEVICE)

CLAP_NAME = "laion/clap-htsat-fused"
clap = ClapModel.from_pretrained(CLAP_NAME).to(DEVICE).eval()
processor = ClapProcessor.from_pretrained(CLAP_NAME)
CLAP_RATE = processor.feature_extractor.sampling_rate
print("CLAP sample rate:", CLAP_RATE)

from synth import audio as audio_lib


def _features(output):
    """transformers versions differ: some return a tensor, some an output object."""
    if isinstance(output, torch.Tensor):
        return output
    for name in ("audio_embeds", "text_embeds", "pooler_output"):
        value = getattr(output, name, None)
        if value is not None:
            return value
    raise TypeError(f"unexpected CLAP output: {type(output)}")


def embed_audio(signals, batch=8):
    rows = []
    for start in range(0, len(signals), batch):
        chunk = [audio_lib.resample(s, 16000, CLAP_RATE) for s in signals[start:start + batch]]
        try:
            inputs = processor(audio=chunk, sampling_rate=CLAP_RATE, return_tensors="pt")
        except TypeError:
            inputs = processor(audios=chunk, sampling_rate=CLAP_RATE, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            rows.append(_features(clap.get_audio_features(**inputs)).float().cpu().numpy())
    return np.concatenate(rows)


def embed_text(texts, batch=32):
    rows = []
    for start in range(0, len(texts), batch):
        inputs = processor(text=list(texts[start:start + batch]), padding=True, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            rows.append(_features(clap.get_text_features(**inputs)).float().cpu().numpy())
    return np.concatenate(rows)


import numpy, scipy, panns_inference, pyloudnorm, soundfile
from importlib.metadata import version
VERSIONS = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "numpy": numpy.__version__,
    "scipy": scipy.__version__,
    "panns-inference": version("panns-inference"),
    "pyloudnorm": version("pyloudnorm"),
    "soundfile": soundfile.__version__,
    "ffmpeg": subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.split("\n")[0],
}
print(VERSIONS)


## 6. Run K1
Prints progress and ETA. Every stage saves state, so a rerun continues.

In [ ]:
paths = dict(data_dir=DATA_DIR, bank_dir=BANK_DIR, realworld_dir=REALWORLD_DIR, out_dir=OUT_DIR, state_dir=STATE_DIR)
if SMOKE_TEST:
    cfg = k1.K1Config(**paths, seed=SEED, repo_commit=REPO_COMMIT)
else:
    cfg = k1.K1Config.full(**paths, seed=SEED, repo_commit=REPO_COMMIT)

deps = k1.Deps(
    fetch=lambda name, root: download.fetch(download.DATASETS[name], root),
    client=client,
    tagger=tagger,
    embed_audio=embed_audio,
    embed_text=embed_text,
    versions=VERSIONS,
    require_codec=True,
)
metrics = k1.run(cfg, deps, clap_model=CLAP_NAME)


## 7. Report

In [ ]:
print((OUT_DIR / "REPORT.md").read_text())


## 8. Zip the outputs

In [ ]:
archive = shutil.make_archive(str(ZIP_PATH), "zip", root_dir=OUT_DIR)
size_mb = Path(archive).stat().st_size / 1e6
print(f"outputs zipped: {archive} ({size_mb:.1f} MB)")
print("Download this zip from the Output tab and unzip it into kaggle_outputs/K1_data_bank/.")
if SMOKE_TEST:
    print("SMOKE OK: set SMOKE_TEST = False, then Save Version > Save & Run All")
